In [ ]:
#!/usr/bin/env python3"""Run all valid combinations on WIDS dataset (local CSV).Uses feature grouping by null pattern similarity to handle the large numberof high-null features (175/183 columns have nulls)."""import itertoolsimport sysimport ossys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))import pandas as pdimport numpy as npfrom core.GenericDataPipeline import GenericDataPipelinefrom core.RunData import RunPipelinefrom sklearn.metrics import roc_auc_scorefrom sklearn.model_selection import train_test_splitfrom core.seed_utils import SEED, set_all_seedspd.set_option('future.infer_string', False)set_all_seeds()# --- Config ---N_TRIALS = int(sys.argv[1]) if len(sys.argv) > 1 else 10NULL_THRESHOLD = 0.50   # Features with >50% nulls are candidates for extendedJACCARD_THRESHOLD = 0.95  # Group features with similar null patternsMAX_GROUPS = 5           # Max number of groups to enumerate (2^5 = 32 combos)MIN_GROUP_PCT = 0.02     # Min 2% of total population per test grouppipeline = GenericDataPipeline()# --- Load data ---csv_path = os.path.join(os.path.dirname(__file__), '..', 'datasets', 'WIDS.csv')df = pd.read_csv(csv_path, na_values=['NA'])# Drop ID columnscolumns_to_drop = ['encounter_id', 'patient_id', 'hospital_id']df = df.drop(columns=columns_to_drop)df = pipeline.preprocessing(df)label = "hospital_death"df[label] = df[label].astype(int)print(f"Dataset shape (before feature selection): {df.shape}")# --- Feature selection: keep top 50 by importance ---import xgboost as xgbprint("Running quick XGBoost for feature selection...")X_all = df.drop(label, axis=1)selector = xgb.XGBClassifier(n_estimators=200, max_depth=6,    window_size=100,    drift_threshold=0.05, learning_rate=0.1,                              tree_method='hist', random_state=SEED, eval_metric='auc')selector.fit(X_all, df[label], verbose=False)imp = pd.Series(selector.feature_importances_, index=X_all.columns).sort_values(ascending=False)top_features = set(imp.head(50).index.tolist())keep_features = top_features | {label}drop_cols = [c for c in df.columns if c not in keep_features]df = df.drop(columns=drop_cols)print(f"Feature selection: kept top {len(top_features)} features by importance")print(f"Dropped {len(drop_cols)} features")print(f"Dataset shape: {df.shape}")print(f"N_TRIALS per model: {N_TRIALS}")print(f"Target distribution:\n{df[label].value_counts()}")# --- OCDS Feature Comparison ---print(f"\n{'='*120}")print("OCDS FEATURE COMPARISON")print(f"{'='*120}")from OCDS.ocds_feature_comparison import OCDSFeatureComparison# Define extended features (fill in the list with feature names)extended_features = []  # TODO: Add extended feature names hereprint(f"Extended features: {extended_features}")print(f"Running OCDS comparison...")# Initialize and run OCDS comparisonocds_comparison = OCDSFeatureComparison(    df=df.copy(),    features_extended=extended_features,    target_col=label,    n_estimators=10,        max_depth=6,    window_size=100,    drift_threshold=0.05,                )# Run the comparisonresult = ocds_comparison.run()print(f"\n{'='*120}")print("OCDS SUMMARY")print(f"{'='*120}")print(f"AUC (Basic features only):  {result['auc_without_extended']:.4f}")print(f"AUC (All features):         {result['auc_with_all']:.4f}")print(f"Difference:                 {result['difference']:+.4f}")print(f"{'='*120}")print("\nOCDS Analysis Complete!")